# XGBoost vs SPLIT vs RESPLIT vs PRAXIS Comparison

Compare four algorithms on airline passenger satisfaction dataset:
- **XGBoost**: Black-box ensemble baseline
- **SPLIT**: Glass-box single optimal tree
- **RESPLIT**: Rashomon set enumeration (command-line)
- **PRAXIS**: Proxy-based Rashomon enumeration

## Part 1: XGBoost and SPLIT Training

In [18]:
import dimex as dx
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SELECTED_FEATURES = [
    'Online_boarding', 'Type_of_Travel_Personal Travel', 'Class_Eco',
    'Inflight_wifi_service', 'On_board_service', 'Customer_Type_disloyal Customer',
    'Inflight_entertainment', 'Checkin_service', 'Leg_room_service'
]

train_data = pd.read_csv('../airline-passenger-satisfaction/train_clean_encoded_balanced.csv')
x_train = train_data[SELECTED_FEATURES]
y_train = train_data['satisfaction']

In [19]:
# Load test data
test_data = pd.read_csv('../airline-passenger-satisfaction/test_clean_encoded.csv')
x_test = test_data[SELECTED_FEATURES]
y_test = test_data['satisfaction']

results = []
SPLIT_REG = 0.005

# 1. Train XGBoost
print("Training XGBoost...")
xgb_model, xgb_size, xgb_runtime = dx.train_xgb(x_train, y_train)
xgb_pred, _ = dx.prediction_xgb(xgb_model, x_test, y_test)
xgb_loss = (xgb_pred != y_test).sum() / len(y_test)
results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, xgb_pred),
    'Precision': precision_score(y_test, xgb_pred),
    'Recall': recall_score(y_test, xgb_pred),
    'F1': f1_score(y_test, xgb_pred),
    'Loss': round(xgb_loss, 6),
    'Size': f"{xgb_size['trees']} trees, {xgb_size['leaves']} leaves",
    'Runtime (s)': round(xgb_runtime, 3)
})

# 2. Train SPLIT
print("Training SPLIT...")
split_model, split_tree, split_meta = dx.train_split(x_train, y_train, lookahead=2, full_depth=6, reg=SPLIT_REG)
split_pred, _ = dx.prediction_split(split_model, x_test, y_test)
split_loss = (split_pred != y_test).sum() / len(y_test) + split_meta['leaves'] * SPLIT_REG
results.append({
    'Model': 'SPLIT',
    'Accuracy': accuracy_score(y_test, split_pred),
    'Precision': precision_score(y_test, split_pred),
    'Recall': recall_score(y_test, split_pred),
    'F1': f1_score(y_test, split_pred),
    'Loss': round(split_loss, 6),
    'Size': f"{split_meta['leaves']} leaves",
    'Runtime (s)': round(split_meta['runtime'], 3)
})

display(pd.DataFrame(results))

Training XGBoost...
Training SPLIT...


,Model,Accuracy,Precision,Recall,F1,Loss,Size,Runtime (s)
0,XGBoost,0.929209,0.905686,0.936208,0.920694,0.070791,"100 trees, 741 leaves",0.305
1,SPLIT,0.919090,0.899225,0.918610,0.908814,0.120910,8 leaves,5.145


In [20]:
import json

with open('../results/results1.json', 'w') as f:
    json.dump(results, f, indent=4)

## Part 2: Load RESPLIT Results (from command-line execution)

Run `python run_resplit.py` from the terminal first, then execute this cell.

In [21]:
import dimex as dx
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json

with open('../results/results1.json', 'r') as f:
    results = json.load(f)

for entry in results:
    entry['Rashomon_Set_Size'] = 'N/A'
    entry['Peak_Memory (MB)'] = 'N/A'

with open('../results/results2.json', 'r') as f:
    results2 = json.load(f)

for entry in results2:
    results.append({
        'Model': entry['Model'],
        'Accuracy': entry['Test_Accuracy'],
        'Precision': entry['Test_Precision'],
        'Recall': entry['Test_Recall'],
        'F1': entry['Test_F1'],
        'Loss': entry['Train_Loss'],
        'Size': f"{entry['Num_Leaves']} leaves",
        'Runtime (s)': entry['Runtime (s)'],
        'Rashomon_Set_Size': entry['Rashomon_Set_Size'],
        'Peak_Memory (MB)': entry['Peak_Memory (MB)']
    })

display(pd.DataFrame(results))

,Model,Accuracy,Precision,Recall,F1,Loss,Size,Runtime (s),Rashomon_Set_Size,Peak_Memory (MB)
0,XGBoost,0.929209,0.905686,0.936208,0.920694,0.070791,"100 trees, 741 leaves",0.305,N/A,N/A
1,SPLIT,0.919090,0.899225,0.918610,0.908814,0.120910,8 leaves,5.145,N/A,N/A
2,RESPLIT (Best Tree),0.673425,0.582637,0.902332,0.708072,0.331557,5 leaves,34.521,6,19.32


In [22]:
with open('../results/results3.json', 'w') as f:
    json.dump(results, f, indent=4)

## Part 3: PRAXIS Training

In [23]:
import time
import tracemalloc
from praxis import PRAXIS, ThresholdGuessBinarizer
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json

SELECTED_FEATURES = [
    'Online_boarding', 'Type_of_Travel_Personal Travel', 'Class_Eco',
    'Inflight_wifi_service', 'On_board_service', 'Customer_Type_disloyal Customer',
    'Inflight_entertainment', 'Checkin_service', 'Leg_room_service'
]

train_data = pd.read_csv('../airline-passenger-satisfaction/train_clean_encoded_balanced.csv')
x_train = train_data[SELECTED_FEATURES]
y_train = train_data['satisfaction']

test_data = pd.read_csv('../airline-passenger-satisfaction/test_clean_encoded.csv')
x_test = test_data[SELECTED_FEATURES]
y_test = test_data['satisfaction']


# Binarize features (PRAXIS requires binary {0,1} input)
binarizer = ThresholdGuessBinarizer()
x_train_bin = binarizer.fit_transform(x_train, y_train)
x_test_bin = binarizer.transform(x_test)

print("Training PRAXIS...")
tracemalloc.start()
start_time = time.perf_counter()
praxis_model = PRAXIS()
praxis_model.fit(x_train_bin, y_train, lambda_reg=0.005, depth_budget=5, rashomon_mult=0.01, lookahead_k=1)
total_runtime = time.perf_counter() - start_time
_, peak_memory = tracemalloc.get_traced_memory()
tracemalloc.stop()
peak_memory_mb = peak_memory / 1024 / 1024

n_trees = praxis_model.count_trees()
print(f"Rashomon set size: {n_trees} trees")
print(f"Runtime: {total_runtime:.3f}s | Peak memory: {peak_memory_mb:.2f} MB")

best_idx = min(range(n_trees), key=lambda i: praxis_model.get_tree_objective(i))
paths, _ = praxis_model.get_tree_paths(best_idx)
num_leaves = len(paths)
train_pred = praxis_model.get_predictions(best_idx, x_train_bin)
best_loss = (train_pred != y_train.values).sum() / len(y_train) + 0.005 * num_leaves
praxis_pred = praxis_model.get_predictions(best_idx, x_test_bin)

with open('../results/results3.json', 'r') as f:
    results = json.load(f)

results.append({
    'Model': 'PRAXIS (Best Tree)',
    'Accuracy': accuracy_score(y_test, praxis_pred),
    'Precision': precision_score(y_test, praxis_pred),
    'Recall': recall_score(y_test, praxis_pred),
    'F1': f1_score(y_test, praxis_pred),
    'Loss': round(best_loss, 6),
    'Size': f"{num_leaves} leaves",
    'Runtime (s)': round(total_runtime, 3),
    'Rashomon_Set_Size': n_trees,
    'Peak_Memory (MB)': round(peak_memory_mb, 2)
})

display(pd.DataFrame(results))

Training PRAXIS...
Rashomon set size: 140 trees
Runtime: 0.740s | Peak memory: 6.27 MB


,Model,Accuracy,Precision,Recall,F1,Loss,Size,Runtime (s),Rashomon_Set_Size,Peak_Memory (MB)
0,XGBoost,0.929209,0.905686,0.936208,0.920694,0.070791,"100 trees, 741 leaves",0.305,N/A,N/A
1,SPLIT,0.919090,0.899225,0.918610,0.908814,0.120910,8 leaves,5.145,N/A,N/A
2,RESPLIT (Best Tree),0.673425,0.582637,0.902332,0.708072,0.331557,5 leaves,34.521,6,19.32
3,PRAXIS (Best Tree),0.919090,0.899225,0.918610,0.908814,0.136402,8 leaves,0.740,140,6.27


In [24]:
with open('../results/results4.json', 'w') as f:
    json.dump(results, f, indent=4)

In [25]:
from split._tree import Leaf
import dimex as dx

def get_split_paths(node, path=None):
    if path is None:
        path = []
    if isinstance(node, Leaf):
        return [(path, node.prediction)]
    paths = []
    paths += get_split_paths(node.left_child, path + [(node.feature, False)])
    paths += get_split_paths(node.right_child, path + [(node.feature, True)])
    return paths

split_feat_map = {f['index']: f['name'] for f in dx.binarized_features(split_model)}

print("=== SPLIT TREE ===")
for path_steps, pred in get_split_paths(split_tree):
    conditions = [f"{split_feat_map[idx]}={'YES' if met else 'NO'}" for idx, met in path_steps]
    print(f"IF {' AND '.join(conditions)} → {'satisfied' if pred else 'not satisfied'}")

print("\n=== PRAXIS TREE ===")
praxis_feat_names = binarizer.get_feature_names_out()
praxis_paths, praxis_preds = praxis_model.get_tree_paths(best_idx)
for path, pred in zip(praxis_paths, praxis_preds):
    conditions = [f"{praxis_feat_names[abs(int(s))]}={'YES' if int(s) > 0 else 'NO'}" for s in path]
    print(f"IF {' AND '.join(conditions)} → {'satisfied' if pred else 'not satisfied'}")

print(f"\nPredictions identical: {(split_pred == praxis_pred).all()}")
print(f"Disagreement count: {(split_pred != praxis_pred).sum()} / {len(split_pred)}")

=== SPLIT TREE ===
IF Inflight_wifi_service <= 0.5=NO → satisfied
IF Inflight_wifi_service <= 0.5=YES AND Type_of_Travel_Personal Travel <= 0.5=NO AND Inflight_wifi_service <= 3.5=NO AND Inflight_entertainment <= 3.5=NO AND Online_boarding <= 4.5=NO → not satisfied
IF Inflight_wifi_service <= 0.5=YES AND Type_of_Travel_Personal Travel <= 0.5=NO AND Inflight_wifi_service <= 3.5=NO AND Inflight_entertainment <= 3.5=NO AND Online_boarding <= 4.5=YES → satisfied
IF Inflight_wifi_service <= 0.5=YES AND Type_of_Travel_Personal Travel <= 0.5=NO AND Inflight_wifi_service <= 3.5=NO AND Inflight_entertainment <= 3.5=YES AND Customer_Type_disloyal Customer <= 0.5=NO → satisfied
IF Inflight_wifi_service <= 0.5=YES AND Type_of_Travel_Personal Travel <= 0.5=NO AND Inflight_wifi_service <= 3.5=NO AND Inflight_entertainment <= 3.5=YES AND Customer_Type_disloyal Customer <= 0.5=YES → not satisfied
IF Inflight_wifi_service <= 0.5=YES AND Type_of_Travel_Personal Travel <= 0.5=NO AND Inflight_wifi_service

In [26]:
train_pred_split = split_model.predict(x_train)
train_pred_praxis = praxis_model.get_predictions(best_idx, x_train_bin)

print(f"Test agreement:  {(split_pred == praxis_pred).mean():.6f}")
print(f"Train agreement: {(train_pred_split == train_pred_praxis).mean():.6f}")



Test agreement:  1.000000
Train agreement: 1.000000


In [ ]:
from pystreed import STreeDClassifier

streed_model = STreeDClassifier(
    optimization_task='accuracy',
    max_depth=5,              # matches depth_budget
    cost_complexity=0.1,    # matches lambda_reg
    n_thresholds=5,           # thresholds per continuous feature
    verbose=True
)
streed_model.fit(x_train, y_train)
streed_pred = streed_model.predict(x_test)


Search d =  2 | .................................. |     0 seconds  Solution = 17886
Search d =  3 | .................................. |     1 seconds  Solution = 14103
Search d =  4 | .................................. |    14 seconds  Solution = 11173
Search d =  5 | .................................. |   156 seconds  Solution =  9810
Total time elapsed: 156
	Terminal time: 67.6209
	Merging time: 0
	LB Merging time: 0
	UB Substracting time: 0.178415
	Reconstructing time: 0.027341
Terminal calls: 13874
	Terminal 1 node: 0
	Terminal 2 node: 0
	Terminal 3 node: 13874
Training score:  0.9164352522275414
Tree depth:  5  	Branching nodes:  29


In [30]:
import tracemalloc, time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json

# Re-run with timing and memory tracking
tracemalloc.start()
start_time = time.perf_counter()
streed_model.fit(x_train, y_train)
total_runtime = time.perf_counter() - start_time
_, peak_memory = tracemalloc.get_traced_memory()
tracemalloc.stop()
peak_memory_mb = peak_memory / 1024 / 1024

streed_pred = streed_model.predict(x_test)
streed_train_pred = streed_model.predict(x_train)

num_leaves = streed_model.get_n_leaves()
train_loss = (streed_train_pred != y_train.values).sum() / len(y_train) + 0.005 * num_leaves

with open('../results/results4.json', 'r') as f:
    results = json.load(f)

results.append({
    'Model': 'STreeD',
    'Accuracy': accuracy_score(y_test, streed_pred),
    'Precision': precision_score(y_test, streed_pred),
    'Recall': recall_score(y_test, streed_pred),
    'F1': f1_score(y_test, streed_pred),
    'Loss': round(train_loss, 6),
    'Size': f"{num_leaves} leaves",
    'Runtime (s)': round(total_runtime, 3),
    'Rashomon_Set_Size': 'N/A',
    'Peak_Memory (MB)': round(peak_memory_mb, 2)
})

display(pd.DataFrame(results))

Search d =  2 | .................................. |     0 seconds  Solution = 17886
Search d =  3 | .................................. |     1 seconds  Solution = 14103
Search d =  4 | .................................. |    14 seconds  Solution = 11173
Search d =  5 | .................................. |   156 seconds  Solution =  9810
Total time elapsed: 156
	Terminal time: 67.1503
	Merging time: 0
	LB Merging time: 0
	UB Substracting time: 0.176743
	Reconstructing time: 0.032991
Terminal calls: 13874
	Terminal 1 node: 0
	Terminal 2 node: 0
	Terminal 3 node: 13874
Training score:  0.9164352522275414
Tree depth:  5  	Branching nodes:  29


,Model,Accuracy,Precision,Recall,F1,Loss,Size,Runtime (s),Rashomon_Set_Size,Peak_Memory (MB)
0,XGBoost,0.929209,0.905686,0.936208,0.920694,0.070791,"100 trees, 741 leaves",0.305,N/A,N/A
1,SPLIT,0.919090,0.899225,0.918610,0.908814,0.120910,8 leaves,5.145,N/A,N/A
2,RESPLIT (Best Tree),0.673425,0.582637,0.902332,0.708072,0.331557,5 leaves,34.521,6,19.32
3,PRAXIS (Best Tree),0.919090,0.899225,0.918610,0.908814,0.136402,8 leaves,0.740,140,6.27
4,STreeD,0.926544,0.903815,0.931808,0.917598,0.233565,30 leaves,159.474,N/A,43.35
